In [1]:
import nltk

# List of packages to download
packages = ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']

for package in packages:
    nltk.download(package)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [21]:
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer,WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer


In [3]:
import pandas as pd

path="../datasets/twitter.csv"
df=pd.read_csv(path,header=None,names=['tweet_id','entity','sentiment','text'])

df.head(10)

,tweet_id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
5,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
6,2402,Borderlands,Positive,So I spent a few hours making something for fu...
7,2402,Borderlands,Positive,So I spent a couple of hours doing something f...
8,2402,Borderlands,Positive,So I spent a few hours doing something for fun...
9,2402,Borderlands,Positive,So I spent a few hours making something for fu...


In [4]:
df.shape

(74682, 4)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   tweet_id   74682 non-null  int64
 1   entity     74682 non-null  str  
 2   sentiment  74682 non-null  str  
 3   text       73996 non-null  str  
dtypes: int64(1), str(3)
memory usage: 2.3 MB


In [6]:
df.isna().sum()

tweet_id       0
entity         0
sentiment      0
text         686
dtype: int64

In [7]:
#Drop rows where 'text' is NaN
df.dropna(subset=['text'],inplace=True)

In [8]:
df.isnull().sum()

tweet_id     0
entity       0
sentiment    0
text         0
dtype: int64

In [9]:
# Reset the index (good practice after dropping rows)
df.reset_index(drop=True,inplace=True)

In [10]:
df.describe

<bound method NDFrame.describe of        tweet_id       entity sentiment  \
0          2401  Borderlands  Positive   
1          2401  Borderlands  Positive   
2          2401  Borderlands  Positive   
3          2401  Borderlands  Positive   
4          2401  Borderlands  Positive   
...         ...          ...       ...   
73991      9200       Nvidia  Positive   
73992      9200       Nvidia  Positive   
73993      9200       Nvidia  Positive   
73994      9200       Nvidia  Positive   
73995      9200       Nvidia  Positive   

                                                    text  
0      im getting on borderlands and i will murder yo...  
1      I am coming to the borders and I will kill you...  
2      im getting on borderlands and i will kill you ...  
3      im coming on borderlands and i will murder you...  
4      im getting on borderlands 2 and i will murder ...  
...                                                  ...  
73991  Just realized that the Windows partition 

In [11]:
df.columns

Index(['tweet_id', 'entity', 'sentiment', 'text'], dtype='str')

In [12]:
df["sentiment"].value_counts()

sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64

In [13]:
import re
def clean_text(text):
    text=re.sub(r"https?://\S+|www\.\S+"," ",text)
    text=re.sub(r"@\w+"," ",text)
    text=re.sub(r"#"," ",text)
    text=re.sub(r"&\w+"," ",text)
    text=re.sub(r"\d+"," ",text)
    text=re.sub(r"^[a-z\s]"," ",text)
    text=re.sub(r"\s+"," ",text).strip()
    
    return text

In [14]:
df["text"]=df["text"].apply(clean_text)

In [15]:
df["tokens"]=df["text"].apply(word_tokenize)

,tweet_id,entity,sentiment,text,tokens
0,2401,Borderlands,Positive,m getting on borderlands and i will murder you...,"[m, getting, on, borderlands, and, i, will, mu..."
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,"[I, am, coming, to, the, borders, and, I, will..."
2,2401,Borderlands,Positive,"m getting on borderlands and i will kill you all,","[m, getting, on, borderlands, and, i, will, ki..."
3,2401,Borderlands,Positive,m coming on borderlands and i will murder you ...,"[m, coming, on, borderlands, and, i, will, mur..."
4,2401,Borderlands,Positive,m getting on borderlands and i will murder you...,"[m, getting, on, borderlands, and, i, will, mu..."
...,...,...,...,...,...
73991,9200,Nvidia,Positive,Just realized that the Windows partition of my...,"[Just, realized, that, the, Windows, partition..."
73992,9200,Nvidia,Positive,Just realized that my Mac window partition is ...,"[Just, realized, that, my, Mac, window, partit..."
73993,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...,"[Just, realized, the, windows, partition, of, ..."
73994,9200,Nvidia,Positive,Just realized between the windows partition of...,"[Just, realized, between, the, windows, partit..."


In [ ]:
lemmatizer= WordNetLemmatizer()

# Lemmatize each token
df["tokens"]=df["tokens"].apply(lambda toks: [lemmatizer.lemmatize(t) for t in toks])

# Join the tokens back into a sentence
df['tokens']=df["tokens"]=df["tokens"].apply(lambda toks:"",join(toks))

In [ ]:
df.head()

In [ ]:
tfidf=TfidfVectorizer(max_features=5000)
X_tfidf=tfidf.fit_transform(df["tokens"])